In [11]:
# Desativando os avisos no notebook para manter células de saída limpas
import warnings
warnings.filterwarnings('ignore')

# Importação das bibliotecas necessárias
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2 as cv
import yaml
from PIL import Image
from ultralytics import YOLO
from IPython.display import Video

In [9]:
# Verifique o diretório 
os.getcwd()

'c:\\Users\\gusta\\Documents\\Faculdade\\Programa k\\Organizacao Trafego\\organizacao-trafego\\semaforo_inteligente'

In [5]:
path_model = 'runs/detect/detect_vehicle/weights/best.pt'
best_model = YOLO(path_model)

In [25]:
# Defina o limite para considerar o tráfego como pesado
heavy_traffic_threshold = 10

# Nova resolução do vídeo
nova_largura = 1280
nova_altura = 720

# Fatores de escala
fator_escala_x = nova_largura / 384  # Para o eixo x
fator_escala_y = nova_altura / 640   # Para o eixo y

# Ajustar as coordenadas dos vértices para o primeiro quadrilátero
novos_vertices1 = np.array([(465, 350), (609, 350), (510, 630), (2, 630)], dtype=np.int32)

# Ajustar as coordenadas dos vértices para o segundo quadrilátero
novos_vertices2 = np.array([(678, 350), (815, 350), (1203, 630), (743, 630)], dtype=np.int32)

# Defina o intervalo vertical para o limite de fatia e pista
x1, x2 = 325, 635
lane_threshold = 609

# Defina as posições das anotações de texto na imagem
text_position_left_lane = (10, 50)
text_position_right_lane = (820, 50)
intensity_position_left_lane = (10, 100)
intensity_position_right_lane = (820, 100)

# Defina fonte, escala e cores para as anotaçõess
font = cv.FONT_HERSHEY_SIMPLEX
font_scale = 1
font_color = (255, 255, 255)    # White color for text
background_color = (0, 0, 255)  # Red background for text

# Abra o vídeo
cap = cv.VideoCapture('cars3.mp4')

# Obtenha as dimensões reais do vídeo
video_width = cap.get(cv.CAP_PROP_FRAME_WIDTH)
video_height = cap.get(cv.CAP_PROP_FRAME_HEIGHT)

frame_count = 0
SKIP_FRAMES = 2

# Definindo a largura e a altura dos frames
LARGURA_FRAME = 640
ALTURA_FRAME = 480

LIMIAR_CONFIANCA = 0.4 # Limiar de confiança

while cap.isOpened():
        # Capturando frame a frame
        ret, frame = cap.read()

        if not ret:
            print("FIM!")
            break

        frame_count += 1
        if frame_count % SKIP_FRAMES != 0:
            continue

        # Redimensionando o frame
        #frame = cv.resize(frame, (LARGURA_FRAME, ALTURA_FRAME))

        # Realizando a detecção de objetos no frame
        results = best_model.predict(source=[frame], conf=LIMIAR_CONFIANCA, save=False, iou=0.70, imgsz=640)
        processed_frame = results[0].plot(line_width=1)

        # Restaure as partes superiores e inferiores originais da moldura
        processed_frame[:x1, :] = frame[:x1, :].copy()
        processed_frame[x2:, :] = frame[x2:, :].copy()

        # Desenhe os quadriláteros no quadro processado
        cv.polylines(processed_frame, [novos_vertices1], isClosed=True, color=(0, 255, 0), thickness=2)
        cv.polylines(processed_frame, [novos_vertices2], isClosed=True, color=(255, 0, 0), thickness=2)

        # Recuperar as caixas delimitadoras dos resultados
        bounding_boxes = results[0].boxes

        # Inicialize contadores para veículos em cada faixa
        vehicles_in_left_lane = 0
        vehicles_in_right_lane = 0

        # Percorra cada caixa delimitadora para contar veículos em cada faixa
        for box in bounding_boxes.xyxy:
            # Verifique se o veículo está na faixa da esquerda com base na coordenada x da caixa delimitadora
            if box[0] < lane_threshold:
                vehicles_in_left_lane += 1
            else:
                vehicles_in_right_lane += 1

        # Determine a intensidade do tráfego para a faixa da esquerda
        traffic_intensity_left = "Pesado" if vehicles_in_left_lane > heavy_traffic_threshold else "Leve"
        # Determine a intensidade do tráfego para a faixa da direita
        traffic_intensity_right = "Pesado" if vehicles_in_right_lane > heavy_traffic_threshold else "Leve"


        # Adicione um retângulo de fundo para a contagem de veículos na faixa esquerda
        #cv.rectangle(processed_frame, (text_position_left_lane[0]-10, text_position_left_lane[1] - 25),
        #              (text_position_left_lane[0] + 460, text_position_left_lane[1] + 10), background_color, -1)

        # Adicione o texto de contagem de veículos no topo do retângulo da faixa da esquerda
        #cv.putText(processed_frame, f'Veículos na faixa da esquerda: {vehicles_in_left_lane}', text_position_left_lane,
        #            font, font_scale, font_color, 2, cv.LINE_AA)

        # Adicione um retângulo de fundo para a intensidade do tráfego na faixa da esquerda
        #cv.rectangle(processed_frame, (intensity_position_left_lane[0]-10, intensity_position_left_lane[1] - 25),
        #              (intensity_position_left_lane[0] + 460, intensity_position_left_lane[1] + 10), background_color, -1)

        # Adicione o texto de intensidade de tráfego no topo do retângulo para a LAN esquerda
        #cv.putText(processed_frame, f'Intensidade de tráfego: {traffic_intensity_left}', intensity_position_left_lane,
        #            font, font_scale, font_color, 2, cv.LINE_AA)

        # Adicione um retângulo de fundo para o número de veículos na faixa da direita
        #cv.rectangle(processed_frame, (text_position_right_lane[0]-10, text_position_right_lane[1] - 25),
        #              (text_position_right_lane[0] + 460, text_position_right_lane[1] + 10), background_color, -1)

        # Adicione o texto da contagem de veículos no topo do retângulo da faixa da direita
        #cv.putText(processed_frame, f'Veículos na faixa da direita: {vehicles_in_right_lane}', text_position_right_lane,
        #            font, font_scale, font_color, 2, cv.LINE_AA)

        # Adicione um retângulo de fundo para a intensidade de tráfego da faixa direita
        #cv.rectangle(processed_frame, (intensity_position_right_lane[0]-10, intensity_position_right_lane[1] - 25),
        #              (intensity_position_right_lane[0] + 460, intensity_position_right_lane[1] + 10), background_color, -1)

        # Adicione o texto de intensidade de tráfego no topo do retângulo da faixa da direita
        #cv.putText(processed_frame, f'Intensidade de tráfego: {traffic_intensity_right}', intensity_position_right_lane,
        #            font, font_scale, font_color, 2, cv.LINE_AA)

        # Remova o comentário das três linhas a seguir se estiver executando este código em uma máquina local para visualizar os resultados do processamento em tempo real
        cv.imshow('Real-time Analysis', processed_frame)
        if cv.waitKey(1) & 0xFF == ord('q'):  # Pressione Q no teclado para sair do loop
            break

cap.release() # Libera a captura de vídeo
cv.destroyAllWindows() # Fecha todas as janelas


0: 384x640 2 vehicles, 182.4ms
Speed: 6.0ms preprocess, 182.4ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 vehicles, 220.2ms
Speed: 4.9ms preprocess, 220.2ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 vehicles, 176.8ms
Speed: 3.6ms preprocess, 176.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 vehicles, 142.2ms
Speed: 2.0ms preprocess, 142.2ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 vehicles, 162.1ms
Speed: 3.0ms preprocess, 162.1ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 vehicles, 146.4ms
Speed: 3.0ms preprocess, 146.4ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 vehicles, 133.8ms
Speed: 2.0ms preprocess, 133.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 vehicles, 125.8ms
Speed: 2.0ms preprocess, 125.8ms inference, 1.0ms postproc